#  K-Means Clustering — Edición Pokémon

**Objetivo:** Aprender a agrupar datos sin etiquetas usando el algoritmo K-Means, aplicado a un mundo que ya conocemos: el universo Pokémon.

---

| Sección | Contenido |
|---|---|
| **Parte 1 — Demo** | Hábitats Pokémon con `make_blobs` |
| **Parte 2 — Laboratorio** | Método del Codo + Dataset de Bayas Pokémon |

## 📦 Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
print('✅ Librerías cargadas correctamente')

---
# 🎓 PARTE 1 — Descubriendo Hábitats Pokémon

Imagina que eres un investigador Pokémon con un sensor que captura la posición de cientos de Pokémon en la región.  
**No sabes de qué tipo son**, pero notas que se concentran en zonas. K-Means te ayudará a descubrir esas zonas automáticamente.

 **Pregunta clave:** ¿Puede un algoritmo encontrar grupos en datos que nunca ha visto etiquetados?

### 🗺️ Paso 1 — Generar los datos: 4 Hábitats Pokémon

In [ ]:
# Generamos 4 nubes de puntos que simulan hábitats
# make_blobs: genera clusters gaussianos
X, y_true = make_blobs(
    n_samples=300,       # 300 avistamientos de Pokémon
    centers=4,           # 4 hábitats distintos
    cluster_std=0.9,     # dispersión dentro de cada hábitat
    random_state=RANDOM_STATE
)

print(f'Forma del dataset: {X.shape}')
print(f'Variables: coordenada X (longitud), coordenada Y (latitud)')
print(f'Primeras 5 observaciones:\n{X[:5]}')

In [ ]:
# Visualizamos los datos SIN etiquetas — así los ve K-Means
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], color='gray', alpha=0.5, s=40, edgecolors='white', linewidths=0.5)
plt.title('🗺️ Avistamientos de Pokémon en la Región\n(Sin etiquetas — datos crudos)', fontsize=14)
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print('⚠️  El algoritmo NO tiene acceso a y_true. Tiene que descubrir los grupos solo.')

### ⚙️ Paso 2 — Implementar K-Means con sklearn

In [ ]:
# ── Hiperparámetros de KMeans ──────────────────────────────────────────────
# n_clusters : número de grupos a encontrar (K)
# init       : estrategia de inicialización de centroides
#              'k-means++' elige centroides alejados entre sí → mejor convergencia
# n_init     : veces que se reinicia el algoritmo (guarda el mejor resultado)
# max_iter   : máximo de iteraciones por ejecución
# random_state: semilla para reproducibilidad
# ──────────────────────────────────────────────────────────────────────────
kmeans = KMeans(
    n_clusters=4,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=RANDOM_STATE
)

# .fit() ejecuta el algoritmo completo:
#   1. Inicializa K centroides
#   2. Asigna cada punto al centroide más cercano
#   3. Recalcula centroides como media del cluster
#   4. Repite 2-3 hasta convergencia
kmeans.fit(X)

print('✅ K-Means entrenado')
print(f'   Iteraciones hasta convergencia : {kmeans.n_iter_}')
print(f'   Etiquetas asignadas (primeras 10): {kmeans.labels_[:10]}')

In [ ]:
# Extraemos la información aprendida por el modelo
labels     = kmeans.labels_           # cluster asignado a cada punto
centroids  = kmeans.cluster_centers_  # coordenadas de cada centroide

print('📍 Coordenadas de los 4 centroides (hábitats detectados):')
habitats = ['🌊 Acuático', '🌋 Volcánico', '🌿 Selvático', '🏔️ Montañoso']
for i, (centroid, name) in enumerate(zip(centroids, habitats)):
    print(f'   Cluster {i} — {name}: ({centroid[0]:.2f}, {centroid[1]:.2f})')

### 🎨 Paso 3 — Visualizar los Clusters y Centroides

In [ ]:
COLORES  = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']  # azul, rojo, verde, naranja
HABITATS = ['🌊 Acuático', '🌋 Volcánico', '🌿 Selvático', '🏔️ Montañoso']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Izquierda: datos reales (y_true) ──────────────────────────────────────
ax = axes[0]
for k in range(4):
    mask = (y_true == k)
    ax.scatter(X[mask, 0], X[mask, 1],
               color=COLORES[k], alpha=0.6, s=40,
               edgecolors='white', linewidths=0.4,
               label=HABITATS[k])
ax.set_title('✅ Etiquetas REALES\n(Solo para comparar — K-Means no las ve)', fontsize=12)
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.4)

# ── Derecha: clusters predichos por K-Means ────────────────────────────────
ax = axes[1]
for k in range(4):
    mask = (labels == k)
    ax.scatter(X[mask, 0], X[mask, 1],
               color=COLORES[k], alpha=0.6, s=40,
               edgecolors='white', linewidths=0.4,
               label=f'Cluster {k}')

# Marcamos los centroides con una 'X' grande y visible
ax.scatter(centroids[:, 0], centroids[:, 1],
           marker='X', s=250, c='black',
           zorder=5, edgecolors='white', linewidths=1.5,
           label='Centroides')

# Anotamos el número de cada centroide
for i, centroid in enumerate(centroids):
    ax.annotate(f' C{i}', xy=centroid, fontsize=11, fontweight='bold', color='black')

ax.set_title('🤖 Clusters predichos por K-Means\n(X = centroides aprendidos)', fontsize=12)
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('K-Means Clustering — Hábitats Pokémon', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 📐 Paso 4 — Inercia (SSE): ¿Qué tan compactos son los clusters?

In [ ]:
# La Inercia (también llamada SSE — Sum of Squared Errors) mide
# la suma de distancias al cuadrado de cada punto a su centroide.
#
#   Inercia baja  → puntos muy cercanos a sus centroides (clusters compactos) ✅
#   Inercia alta  → puntos dispersos, clusters poco definidos              ❌
#
# sklearn la calcula y almacena en .inertia_ tras el fit()

inertia = kmeans.inertia_
print(f'📊 Inercia (SSE) con K=4: {inertia:.2f}')
print()
print('Cálculo manual para verificar:')
sse_manual = 0
for k in range(4):
    puntos_cluster = X[labels == k]
    centroide_k    = centroids[k]
    distancias_sq  = np.sum((puntos_cluster - centroide_k) ** 2, axis=1)
    sse_cluster    = np.sum(distancias_sq)
    sse_manual    += sse_cluster
    print(f'   Cluster {k}: SSE parcial = {sse_cluster:.2f}  ({len(puntos_cluster)} puntos)')

print(f'   ─────────────────────────────')
print(f'   SSE total manual : {sse_manual:.2f}')
print(f'   kmeans.inertia_  : {inertia:.2f}')
print(f'   ¿Coinciden? {np.isclose(sse_manual, inertia)}')

### 💡 Resumen Parte 1

| Concepto | Definición rápida |
|---|---|
| **K** | Número de clusters que elige el usuario |
| **Centroide** | Punto medio de cada cluster (aprende en el entrenamiento) |
| **Asignación** | Cada punto va al centroide más cercano (distancia Euclídea) |
| **Inercia (SSE)** | Suma de distancias² punto→centroide. Menor es mejor |
| **k-means++** | Inicialización inteligente: centroides alejados entre sí → convergencia más estable |

> **Limitación clave:** K-Means necesita que le digas cuántos clusters quieres.  
> ¿Cómo elegir el K óptimo? → **Método del Codo** (tu turno en la Parte 2 👇)

---
# 🧪 PARTE 2 — Laboratorio

> **Instrucciones:** Completa las celdas marcadas con `# TODO`.  
> Cada TODO indica exactamente qué debes escribir. ¡Usa lo que aprendiste en la demo!

## 🗡️ Tarea 1 — El Método del Codo

**Contexto:** Volvemos a los datos de Hábitats Pokémon (`X`), pero esta vez **fingimos no saber que hay 4 grupos**.  
El Método del Codo nos ayuda a elegir el K óptimo graficando cómo cae la inercia al aumentar K.

**¿Por qué funciona?**  
- Al aumentar K, la inercia siempre baja (más centroides = puntos más cerca de su centroide).  
- Pero hay un punto donde añadir más K ya no mejora mucho → el **"codo"** de la curva.  
- Ese codo es nuestro K óptimo.

In [ ]:
# ── Tarea 1: Método del Codo ───────────────────────────────────────────────
# Calcula la inercia para K = 1, 2, 3, ..., 10 y guárdalas en una lista.
# Luego grafica K (eje X) vs Inercia (eje Y).
# ──────────────────────────────────────────────────────────────────────────

inercias = []  # lista donde guardaremos la inercia para cada K
rango_k  = range(1, 11)  # K del 1 al 10

for k in rango_k:
    # TODO 1: Crea un KMeans con n_clusters=k, init='k-means++',
    #         n_init=10, random_state=RANDOM_STATE
    modelo = None  # ← reemplaza None con tu código

    # TODO 2: Entrena el modelo con los datos X
    # ...

    # TODO 3: Extrae la inercia del modelo entrenado y añádela a la lista
    # inercias.append(...)
    pass

print('Inercias calculadas:', inercias)

In [ ]:
# ── Tarea 1 (continuación): Graficar el Método del Codo ───────────────────

plt.figure(figsize=(9, 5))

# TODO 4: Grafica rango_k en el eje X e inercias en el eje Y
#         Usa plt.plot() con marcadores ('o-') y un color que te guste
# ...

# TODO 5: Añade una línea vertical punteada en el K óptimo que detectes
#         Pista: plt.axvline(x=???, color='red', linestyle='--', label='K óptimo')
# ...

plt.title('📐 Método del Codo — ¿Cuántos Hábitats Pokémon existen?', fontsize=13)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (SSE)')
plt.xticks(list(rango_k))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# TODO 6: Escribe aquí tu conclusión
# ¿En qué valor de K ves el "codo"? ¿Coincide con los 4 hábitats reales?
print('📝 Mi conclusión: ...')

---
## 🍓 Tarea 2 — Dataset de Bayas Pokémon

**Historia:** El Professor Oak ha recolectado **210 muestras de bayas silvestres** de toda la región.  
Midió 7 características físicas de cada baya, pero **no las clasificó** porque no tiene tiempo.  
Tu misión: usar K-Means para descubrir **cuántos tipos de bayas existen**.

*(Dataset basado en el clásico Seeds Dataset de UCI — trigo → bayas Pokémon)*

**Características medidas:**

| Columna | Descripción |
|---|---|
| `area` | Área de la baya (mm²) |
| `perimetro` | Perímetro (mm) |
| `compacidad` | Índice de forma circular |
| `longitud_nucleo` | Longitud del núcleo (mm) |
| `ancho_nucleo` | Ancho del núcleo (mm) |
| `coef_asimetria` | Coeficiente de asimetría |
| `longitud_surco` | Longitud del surco central (mm) |

In [ ]:
# Cargamos el dataset de Bayas Pokémon (Seeds Dataset de UCI)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt'

columnas = ['area', 'perimetro', 'compacidad',
            'longitud_nucleo', 'ancho_nucleo',
            'coef_asimetria', 'longitud_surco', 'tipo_real']

df_bayas = pd.read_csv(url, sep=r'\s+', header=None, names=columnas)

# Separamos features y la etiqueta real (solo para comparar al final)
X_bayas   = df_bayas.drop(columns='tipo_real').values
y_bayas   = df_bayas['tipo_real'].values

print(f'Dataset de Bayas Pokémon cargado: {df_bayas.shape[0]} muestras, {X_bayas.shape[1]} features')
df_bayas.drop(columns='tipo_real').describe().round(2)

In [ ]:
# ── Tarea 2a: Método del Codo para las Bayas ──────────────────────────────
# Antes de aplicar K-Means, DEBES escalar los datos.
# (Verás POR QUÉ en la sección final — por ahora hazlo sin cuestionarlo)

# TODO 7: Crea un StandardScaler y ajústalo+transfórmalo sobre X_bayas
#         Guarda el resultado en X_bayas_scaled
#         Pista: scaler = StandardScaler() → scaler.fit_transform(...)
X_bayas_scaled = None  # ← reemplaza con tu código

# TODO 8: Calcula la inercia para K = 1 al 10 usando X_bayas_scaled
#         (mismo patrón que en la Tarea 1)
inercias_bayas = []

# for k in range(1, 11):
#     ...

# TODO 9: Grafica el Método del Codo para las bayas
plt.figure(figsize=(9, 5))
# ...
plt.title('🍓 Método del Codo — ¿Cuántos tipos de Bayas Pokémon hay?', fontsize=13)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (SSE)')
plt.xticks(range(1, 11))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('📝 Según el gráfico, el K óptimo parece ser: ...')

In [ ]:
# ── Tarea 2b: Aplicar K-Means con el K óptimo ─────────────────────────────

# TODO 10: Entrena KMeans con el K que elegiste en la Tarea 2a
#          usando X_bayas_scaled
K_optimo   = 3   # ← ajusta este valor según tu gráfico del codo
kmeans_bayas = None  # ← reemplaza con tu KMeans entrenado

# TODO 11: Obtén las etiquetas predichas y guárdalas en labels_bayas
labels_bayas = None  # ← reemplaza

# Comparación con las etiquetas reales (el dataset tiene 3 tipos)
df_resultado = df_bayas.drop(columns='tipo_real').copy()
df_resultado['cluster_kmeans'] = labels_bayas
df_resultado['tipo_real']      = y_bayas

print('📊 Distribución de tipos reales vs clusters encontrados:')
print(pd.crosstab(df_resultado['tipo_real'], df_resultado['cluster_kmeans'],
                  rownames=['Tipo Real'], colnames=['Cluster K-Means']))

In [ ]:
# ── Tarea 2c (Bonus): Visualizar las Bayas en 2D ──────────────────────────
# Como tenemos 7 dimensiones, usamos solo las 2 más representativas para visualizar

# TODO 12 (Bonus): Grafica un scatter plot de 'area' vs 'longitud_nucleo'
#                  coloreando por cluster_kmeans
#                  Compara con otro scatter coloreado por tipo_real
#                  ¿Se parecen?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
COLORES_B = ['#9B59B6', '#1ABC9C', '#E67E22']

for ax, col_color, titulo in zip(axes,
                                  ['tipo_real', 'cluster_kmeans'],
                                  ['✅ Tipos reales', '🤖 Clusters K-Means']):
    grupos = df_resultado[col_color].unique()
    for g, color in zip(sorted(grupos), COLORES_B):
        mask = df_resultado[col_color] == g
        # TODO: descomenta y completa
        # ax.scatter(df_resultado.loc[mask, 'area'],
        #            df_resultado.loc[mask, 'longitud_nucleo'],
        #            color=color, alpha=0.6, s=50,
        #            edgecolors='white', linewidths=0.4,
        #            label=f'Grupo {g}')
        pass
    ax.set_title(f'Bayas Pokémon — {titulo}', fontsize=12)
    ax.set_xlabel('Área (mm²)')
    ax.set_ylabel('Longitud del Núcleo (mm)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

---
## ⚖️ Sección Final — ¿Por qué StandardScaler es ESENCIAL antes de K-Means?

K-Means mide **distancias**. Si una variable tiene valores en miles y otra en decimales,  
la primera dominará completamente el cálculo, **aunque no sea más importante**.

StandardScaler transforma cada variable a **media=0, desviación estándar=1**,  
poniendo todas en el mismo campo de juego.

In [ ]:
# ── Demostración: efecto de no escalar ────────────────────────────────────
np.random.seed(42)
n = 100

# Dataset Pokémon simulado: Peso (kg, escala grande) y Velocidad (0-1, escala pequeña)
peso      = np.concatenate([np.random.normal(8,   1,   n),   # Pokémon ligeros
                             np.random.normal(80,  5,   n)])  # Pokémon pesados
velocidad = np.concatenate([np.random.normal(0.9, 0.05, n),  # rápidos
                             np.random.normal(0.1, 0.05, n)]) # lentos

# Nótese que los ligeros son rápidos y los pesados son lentos → 2 grupos claros
X_demo = np.column_stack([peso, velocidad])

# K-Means SIN escalar
km_sin  = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_demo)

# K-Means CON escalar
scaler  = StandardScaler()
X_scaled_demo = scaler.fit_transform(X_demo)
km_con  = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_scaled_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, labels_plot, X_plot, km_plot, xlabel, ylabel, titulo in zip(
        axes,
        [km_sin.labels_, km_con.labels_],
        [X_demo, X_scaled_demo],
        [km_sin, km_con],
        ['Peso (kg)', 'Peso (estandarizado)'],
        ['Velocidad (0–1)', 'Velocidad (estandarizada)'],
        ['❌ Sin StandardScaler\n(el peso domina — K-Means se confunde)',
         '✅ Con StandardScaler\n(ambas variables pesan igual — clusters correctos)']):

    for k, color in zip([0, 1], ['#3498DB', '#E74C3C']):
        mask = labels_plot == k
        ax.scatter(X_plot[mask, 0], X_plot[mask, 1],
                   color=color, alpha=0.5, s=30, label=f'Cluster {k}')
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('⚖️ Impacto de StandardScaler en K-Means', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualización del efecto del scaler sobre los datos ───────────────────
print('📊 Estadísticas ANTES del escalado:')
df_sin = pd.DataFrame(X_demo, columns=['Peso (kg)', 'Velocidad'])
print(df_sin.describe().round(3))

print('\n📊 Estadísticas DESPUÉS del escalado:')
df_con = pd.DataFrame(X_scaled_demo, columns=['Peso (scaled)', 'Velocidad (scaled)'])
print(df_con.describe().round(3))

print()
print('🔑 Regla de oro:')
print('   Siempre aplica StandardScaler (o MinMaxScaler) antes de K-Means')
print('   cuando tus features tienen diferentes unidades o escalas.')

---
## 🏆 Resumen Final

| Concepto | Lo que aprendiste |
|---|---|
| **K-Means** | Algoritmo iterativo que minimiza la inercia agrupando puntos por cercanía al centroide |
| **Inercia (SSE)** | Métrica de compacidad: suma de distancias² punto→centroide |
| **Método del Codo** | Técnica para elegir K: busca el quiebre en la curva Inercia vs K |
| **k-means++** | Inicialización que evita mínimos locales malos |
| **StandardScaler** | **Obligatorio** cuando features tienen diferentes escalas |

### Próximos pasos
- **DBSCAN**: clustering que encuentra clusters de forma arbitraria y detecta outliers.
- **Clustering Jerárquico**: no necesita K predefinido, produce un dendrograma.
- **PCA + K-Means**: reducir dimensionalidad antes de clustering para datasets de alta dimensión.

> *"El aprendizaje no supervisado es como ser el Professor Oak: explorar el mundo Pokémon sin que nadie te diga qué encontrarás."*